<a href="https://colab.research.google.com/github/manidharc7/isi-2026-group1-property-valuation/blob/main/ISI_Project_Property_Valuation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this once if any library is missing (uncomment the line below)
# !pip install -q scikit-learn pandas numpy matplotlib joblib groq

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("setup ok")

setup ok


In [2]:
df_raw = pd.read_csv("AmesHousing.csv")

print("shape:", df_raw.shape)
print(df_raw.columns.tolist()[:12])

df_raw[["Overall Qual", "Gr Liv Area", "Year Built", "Neighborhood", "SalePrice"]].head()

shape: (2930, 82)
['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config']


,Overall Qual,Gr Liv Area,Year Built,Neighborhood,SalePrice
0,6,1656,1960,NAmes,215000
1,5,896,1961,NAmes,105000
2,6,1329,1958,NAmes,172000
3,7,2110,1968,NAmes,244000
4,5,1629,1997,Gilbert,189900


In [3]:
# --- (1) Drop De Cock's 5 documented outlier sales ---
before = len(df_raw)
df = df_raw[df_raw["Gr Liv Area"] <= 4000].copy()
print(f"dropped {before - len(df)} outliers; rows now {len(df)}")  # expect 5 dropped, 2925 remain

# --- (2) Define the 14-feature subset ---
NUMERIC = [
    "Overall Qual",    # 1-10 quality rating        — strongest price driver
    "Gr Liv Area",     # above-ground living area (sq ft)
    "Total Bsmt SF",   # basement area (sq ft)
    "Garage Cars",     # garage capacity (cars)
    "Garage Area",     # garage size (sq ft)
    "Year Built",      # original build year
    "Year Remod/Add",  # last remodel year
    "Full Bath",       # full bathrooms above ground
    "TotRms AbvGrd",   # total rooms above ground (excludes bathrooms)
    "1st Flr SF",      # first-floor area (sq ft)
    "Lot Area",        # lot / land size (sq ft)
    "Fireplaces",      # number of fireplaces
    "Bedroom AbvGr",   # bedrooms above ground
]

CATEGORICAL = ["Neighborhood"]
TARGET = "SalePrice"
FEATURES = NUMERIC + CATEGORICAL

# --- (3) Build X and y ---
X = df[FEATURES].copy()
y = df[TARGET].copy()

print("X:", X.shape, " y:", y.shape)   # expect X: (2925, 14)  y: (2925,)
print("SalePrice in X?", TARGET in X.columns)  # must be False

dropped 5 outliers; rows now 2925
X: (2925, 14)  y: (2925,)
SalePrice in X? False
